# 02 - Data Cleaning (Merged CICIDS2017 Dataset)

Cleans the merged dataset produced by `datasets/merge_cicids2017_parquet.py`.

**This version is not just a re-run of the old cleaning notebook on new data — the cleaning strategy itself changed.** The old notebook did a blanket `df.dropna()`. On the merged dataset that wipes out **every row**, because `SimillarHTTP` is null in 100% of rows across all 8 source files. The fix:

- Replace infinite values with `NaN` (unchanged from before)
- Fill remaining `NaN` in **feature** columns with `0` instead of dropping the row
- Only drop rows where **`Label`** itself is missing — a missing target is the one thing we can't recover from

If you ever see row count collapse to 0 after a `dropna()` call on this dataset, check for an all-null column like `SimillarHTTP` first.

1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

print("Libraries Imported Successfully!")

2. Load Dataset

In [ ]:
dataset_path = "../datasets/merged/cicids2017_merged.parquet"

df = pd.read_parquet(dataset_path, engine="pyarrow")

# Remove extra spaces from column names
df.columns = df.columns.str.strip()

print("Merged Dataset Loaded Successfully!")

3. Dataset overview

In [ ]:
print("=" * 60)
print("Original Dataset Shape")
print("=" * 60)

print(df.shape)

4. Checking missing values

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

In [ ]:
if "SimillarHTTP" in df.columns:
    null_pct = df["SimillarHTTP"].isnull().mean() * 100
    print(f"SimillarHTTP is {null_pct:.2f}% null -- confirmed all-null column across the merged dataset.")

5. Checking infinite values

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

infinite = np.isinf(numeric_df).sum()

print(infinite[infinite > 0])

6. Replace infinite values with NaN

In [ ]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Infinite values replaced with NaN.")

7. Checking missing values again

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

8. Handle missing values (feature columns)

**This is the key change from the old notebook.** Instead of `df.dropna(inplace=True)`, which would drop 100% of rows because of `SimillarHTTP` alone, feature-column `NaN`s (from the all-null column, and from `Flow Bytes/s` / `Flow Packets/s` divide-by-zero cases on zero-duration flows) are filled with `0`. Zero is a defensible fill here: a null `SimillarHTTP` carries no signal to begin with, and a zero-duration flow legitimately has zero throughput.

In [ ]:
feature_columns = [col for col in df.columns if col != "Label"]

rows_before = df.shape[0]

df[feature_columns] = df[feature_columns].fillna(0)

print(f"Rows Before Fill : {rows_before}")
print(f"Rows After Fill  : {df.shape[0]}  (unchanged -- fillna doesn't drop rows)")

9. Drop rows with a missing Label

In [ ]:
rows_before = df.shape[0]

df.dropna(subset=["Label"], inplace=True)

rows_after = df.shape[0]

print(f"Rows Before : {rows_before}")
print(f"Rows After  : {rows_after}")
print(f"Rows Removed: {rows_before - rows_after}")

10. Remove duplicate rows

In [ ]:
duplicates = df.duplicated().sum()

print(f"Duplicate Rows Before: {duplicates}")

df.drop_duplicates(inplace=True)

print(f"Dataset Shape After Removing Duplicates: {df.shape}")

11. Verify cleaning

In [ ]:
print("=" * 60)

print("Missing Values")
print(df.isnull().sum().sum())

print("=" * 60)

print("Duplicate Rows")
print(df.duplicated().sum())

print("=" * 60)

numeric_df = df.select_dtypes(include=np.number)

print("Infinite Values")
print(np.isinf(numeric_df).sum().sum())

print("=" * 60)

print("Row count sanity check (should be close to original, NOT zero):")
print(df.shape[0])

12. Data types

In [ ]:
df.dtypes

13. Dataset shape after cleaning

In [ ]:
print("Final Dataset Shape")

print(df.shape)

14. Save cleaned dataset

Saved as Parquet rather than CSV — the merged dataset is ~241MB and CSV would both bloat on disk and be much slower to reload in the next notebook / training scripts.

In [ ]:
output_path = "../datasets/processed/cicids2017_cleaned.parquet"

df.to_parquet(output_path, engine="pyarrow", index=False)

print("Cleaned dataset saved successfully!")

15. Final summary

In [ ]:
print("=" * 60)

print("DATA CLEANING SUMMARY")

print("=" * 60)

print(f"Final Shape    : {df.shape}")

print(f"Missing Values : {df.isnull().sum().sum()}")

print(f"Duplicate Rows : {df.duplicated().sum()}")

numeric_df = df.select_dtypes(include=np.number)

print(f"Infinite Values: {np.isinf(numeric_df).sum().sum()}")

print(f"Attack Classes : {df['Label'].nunique()}")

print("\nLabel distribution after cleaning:")
print(df["Label"].value_counts())

print("=" * 60)